In [ ]:
## Proof-of-work with dask:
#  By organizing the brute-force-search into medium-sized computational chunks,
#  the dask library can distribute this over processes on a single computer
#  or over workers rented from a cloud provider.

#  This notebook deploys the proof-of-work distributed over a cluster of 
#  EC2 instances.

In [ ]:
# Dispatches a hash-cracking workload to multiple servers using coiled to orchestrate 
# scheduler + workers on AWS.  

# The (python) client environment must be very closely matched to the scheduler and the workers.
# This is easiest to accomplish by running jupyter via docker:

#  docker run -v $(PWD):/home/jovyan/work   -e NB_UID=$(id -u)  -p 8888:8888  ghcr.io/dask/dask-notebook:latest

# What does this do?  Start docker container image ghcr.io/dask/dask-notebook (a 2Gb download)
# Allow the docker container access to the current working directory, which will appear as $HOME/work 
# Allow the docker container to receive traffic on port 8888, and run the default endpoint, which 
# runs the jupyter server. 

# The coiled workers are running ghcr.io/dask/dask:latest


In [ ]:
# But this docker container with jupyter and dask doesn't have coiled.  
# This command runs in the container, and does not have effect after 
# the container is stopped, so must be run each time you use this docker
# image for the notebook environment.  
!pip install coiled

In [ ]:
# This command reserves the cluster.  It will confirm that you are 
# authenticated, asking you to open a browser window that is logged 
# in to coiled.

# The coiled account must be provisioned with cloud service provider 
# credentials, and you will be billed for usage.  The $0.016/hour 
# servers aren't powerful enough to run docker, so we need at 
# least medium at about $0.05/hr.

# Coiled will shut down the cluster after 20 minutes of inactivity.

import coiled
cluster = coiled.Cluster(n_workers=5, worker_vm_types=["m7a.medium"],
     container="ghcr.io/dask/dask" )

client = cluster.get_client()

In [ ]:
import hashlib
import dask.bag as db
import datetime

'''Attack on sha256+salt using dask bag + coiled for cloud parallelization'''

wordlist = ["the", "The"]

puzzleeasy = [ x.strip() for x in open("PUZZLE-EASY") ] 
puzzlehard = [ x.strip() for x in open("PUZZLE-HARD") ] 
puzzle = [ x.strip() for x in open("PUZZLE") ] 

In [ ]:
#  Easy puzzle

pset = set(puzzleeasy)
Nhash = 1000
N = 10
tgts = range(N)
def check_hash_easy(j):
    for i in range(j*Nhash, (j+1)*Nhash):
        key = "{:09d}".format(i)

        for word in wordlist:
            hash = hashlib.sha256(key.encode("utf-8") + word.encode("utf-8")).hexdigest()
            if hash in pset:
                return (hash, key, word)
    return None

In [ ]:
#  Medium puzzle

pset = set(puzzle)
Nhash = 5000000
N = 200
tgts = range(N)
def check_hash_medium(j):
    for i in range(j*Nhash, (j+1)*Nhash):
        key = "{:09d}".format(i)
        for word in wordlist:
            hash = hashlib.sha256(key.encode("utf-8") + word.encode("utf-8")).hexdigest()
            if hash in pset:
                return (hash, key, word)
    return None

In [ ]:
#  Hard puzzle

pset = set(puzzlehard)
Nhash = 5000000
N = 200   # Changeme!!
tgts = range(N)
def check_hash_hard(j):
    for i in range(j*Nhash, (j+1)*Nhash):
        key = "{:011d}".format(i)
        for word in wordlist:
            hash = hashlib.sha256(key.encode("utf-8") + word.encode("utf-8")).hexdigest()
            if hash in pset:
                return (hash, key, word)
    return None

In [ ]:
def main():
    bag = db.from_sequence(tgts, npartitions=2000)  
    results = bag.map(check_hash).filter(lambda x: x is not None).compute()
    for result in results:
        print(result)

In [ ]:
check_hash = check_hash_easy
start = datetime.datetime.now()
main()
print(datetime.datetime.now()-start, 'seconds')

In [ ]:
check_hash = check_hash_medium

start = datetime.datetime.now()
main()
print(datetime.datetime.now()-start, 'seconds')

In [ ]:
check_hash = check_hash_hard
start = datetime.datetime.now()
main()
print(datetime.datetime.now()-start, 'seconds')